# 53 · Detect unusual counts, checked against numpy

## What this checks

**Detect unusual counts** holds each period of a series of counts, such as visits per day, to a threshold. The
threshold is set from the series' own past. A count above it is a signal. This check covers release 0.3.0.

It offers five methods:

- **EARS C1, C2 and C3** (Hutwagner and others, 2003). Three simple rules that compare a day with the mean and spread
  of the days just before it.
- **CUSUM**, a cumulative sum (Page, 1954). It adds up how far each period is above what was expected, and signals
  when the total passes a limit.
- **EWMA**, an exponentially weighted moving average (Roberts, 1959). It is a running average that gives the most
  weight to the latest periods, and signals when it passes a limit.

A missed signal, or a false one, decides whether a team goes out. So it is worth checking.

## How it is checked

We write each method again from its published definition, and from the settings Fieldwork declares. **numpy**,
Python's library for arrays of numbers, does the arithmetic. **scipy**, Python's library for statistics, gives the
values of the normal curve. No code is shared with Fieldwork.

There are four series, run 24 ways in all:

- **40 made-up days**, with an excess on days 25 to 29. Fieldwork's own tests hold this series to R's surveillance
  package, release 1.25.0. R's values come with it, so this check is a third reading of the same days.
- **The syndromic example.** Gastrointestinal visits per day, 175 days, with a planted outbreak from 13 to 22 April.
- **Three seasonal years.** Made-up weekly counts with a peak every winter. They show what a method without a
  seasonal baseline does with a season.
- **A rare condition.** Made up, with most days 0 and one cluster. It tests baselines that do not vary, and one whose
  mean is 0.

## What should happen

1. **The same periods computed.** A value where Fieldwork has one, and none where it has none, for every measure.
2. **The same numbers.** Expected count, threshold, statistic and limit within 10⁻⁶, relative where above 1.
   Fieldwork's normal-curve values agree with scipy's to about 10⁻⁷.
3. **The same signals.** A period whose statistic is within that margin of its limit is listed, not counted against.
4. **The same refusals.** Where Fieldwork will not compute, neither does this. CUSUM refuses a baseline whose mean is
   0. EWMA refuses one that does not vary.
5. **Against R, on the 40 made-up days.** EARS C1 and C2 thresholds within 10⁻⁶ of R's recorded values, and CUSUM's
   the same. EARS C3 is reported, not judged (see below).

These criteria were fixed before the check was run.

In [ ]:
NEEDED = ["numpy", "scipy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the installed packages.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_53_detect_unusual_counts import run

report = run(Path.cwd() / "fixtures" / "detect-unusual-counts-001.json")
print("agrees:", report["agrees"])
for r in report["runs"]:
    said = f"{r['signals']} signal" if "signals" in r else f"refused: {r['refused']['here']}"
    print(f"  {'ok' if r['agrees'] else 'NO'}  {r['run']}: {said}", *r["differs"][:3])

## The result on record

First recorded on 22 November 2026, against release 0.2.0. Rerun on 26 November 2026, against release 0.3.0:
**Agrees**.

All 24 runs on the four series agree, including two refusals made alike. Against R, C1, C2 and CUSUM give identical
thresholds and signal days. C3 differs on the first day of the rise, as the check said it would.

Running the cell above repeats the check on the same data.

## Against R, on the 40 days

C1, C2 and CUSUM are judged against R. C3 is only reported. The published paper counts the current day in C3, and R
leaves it out. So R cannot signal on the first day of a rise.

In [ ]:
for method, v in report["againstR"].items():
    print(f"{method:8} largest threshold difference {v['largestThresholdDifference']}")
    print(f"         signals here {v['signals']['here']}")
    print(f"         signals in R {v['signals']['R']}")

## What the seasonal series shows

The peak comes each January. None of the methods has a seasonal baseline, and each behaves differently:

- **CUSUM** is held to the first year's mean. It signals for weeks on end in each later winter: 50 of 156 weeks.
- **EWMA** has a limit set by the first year's spread, which the season widens. It signals in no week.
- **EARS** signals on every autumn's rise, each year. An expected rise looks unusual against the weeks just before
  it.

A seasonal baseline would hold each week to the same weeks of past years.

In [ ]:
for r in report["runs"]:
    if r["run"].startswith("seasonal"):
        print(f"{r['run']}: {r.get('signals')} of {r.get('periods')} weeks signal")

## What this does not show

- That a signal is an outbreak. Both sides compute the same statistic from the same baseline.
- That any setting suits a surveillance system. The settings α, h and L decide how many false alarms there are, and
  that is a decision, not a calculation.
- A seasonal baseline, such as Farrington's method or a regression model. Fieldwork has none, and the seasonal series
  shows what follows.
- How Fieldwork reads a table into a series: putting periods in order, and counting a missing period as 0. The check
  starts from the series as Fieldwork read them. Fieldwork's own tests cover that step.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
e = report["environment"]
print("numpy   :", e["numpy"], "· scipy", e["scipy"])
print("runtime :", e["runtime"], "· Pyodide" if e["pyodide"] else "")